# Financial Fraud Detection

## Phase 13 - Real-Time Fraud Prediction

This notebook:

- Connects Spark Streaming to Kafka
- Reads live transactions
- Parses JSON
- Loads the trained Random Forest model
- Predicts fraud in real time

## Imports

In [1]:
import json
import joblib
import pandas as pd

from pyspark.sql import SparkSession

from pyspark.sql.functions import (
    col,
    from_json,
    when,
    log1p
)

from pyspark.sql.types import *

## Layout Cleaning

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", None)

## Create Spark Session

In [3]:
spark = (
    SparkSession.builder
    .appName("RealTimeFraudPrediction")
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.1"
    )
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/11 22:41:19 WARN Utils: Your hostname, AHMADs-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.29.167 instead (on interface en0)
26/07/11 22:41:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/yaseen/Data_Analysis/Project_Files/Financial_Fraud_Detection/venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/yaseen/.ivy2.5.2/cache
The jars for the packages stored in: /Users/yaseen/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a357ae54-acef-4dce-a1b2-c182c769f147;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.0.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.0.1 in central

## Stop ALL running queries

In [4]:
for q in spark.streams.active:
    print("Stopping:", q.id)
    q.stop()

In [5]:
spark.streams.active

[]

## Load Best Model

In [6]:
model = joblib.load("../models/best_model.pkl")

print(model)

RandomForestClassifier(class_weight='balanced', n_jobs=-1, random_state=42)


## Load Metadata

In [7]:
with open("../models/best_model_metadata.json","r") as f:
    metadata = json.load(f)

metadata

{'model_name': 'Random Forest (Class Weights)',
 'filename': 'best_model.pkl',
 'algorithm': 'RandomForestClassifier',
 'accuracy': 0.999489,
 'precision': 0.958333,
 'recall': 0.726316,
 'f1_score': 0.826347,
 'roc_auc': 0.943893,
 'random_state': 42,
 'scikit_learn_version': '1.9.0',
 'saved_on': '2026-07-11 20:13:28'}

## Read Kafka Stream

In [8]:
stream_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers","localhost:9092")
    .option("subscribe","fraud-transactions")
    .load()
)

## Schema

In [9]:
schema = StructType([

    StructField("Time", DoubleType()),

    StructField("V1", DoubleType()),
    StructField("V2", DoubleType()),
    StructField("V3", DoubleType()),
    StructField("V4", DoubleType()),
    StructField("V5", DoubleType()),
    StructField("V6", DoubleType()),
    StructField("V7", DoubleType()),
    StructField("V8", DoubleType()),
    StructField("V9", DoubleType()),
    StructField("V10", DoubleType()),
    StructField("V11", DoubleType()),
    StructField("V12", DoubleType()),
    StructField("V13", DoubleType()),
    StructField("V14", DoubleType()),
    StructField("V15", DoubleType()),
    StructField("V16", DoubleType()),
    StructField("V17", DoubleType()),
    StructField("V18", DoubleType()),
    StructField("V19", DoubleType()),
    StructField("V20", DoubleType()),
    StructField("V21", DoubleType()),
    StructField("V22", DoubleType()),
    StructField("V23", DoubleType()),
    StructField("V24", DoubleType()),
    StructField("V25", DoubleType()),
    StructField("V26", DoubleType()),
    StructField("V27", DoubleType()),
    StructField("V28", DoubleType()),

    StructField("Amount", DoubleType()),
    StructField("transaction_id", StringType(), True),
    StructField("event_timestamp", StringType(), True),
    StructField("Class", DoubleType())
])

## Parse JSON

In [10]:
parsed_df = (
    stream_df
    .selectExpr("CAST(value AS STRING)")
    .select(
        from_json(
            col("value"),
            schema
        ).alias("data")
    )
    .select("data.*")
)

## Verify Schema

In [11]:
parsed_df.printSchema()

root
 |-- Time: double (nullable = true)
 |-- V1: double (nullable = true)
 |-- V2: double (nullable = true)
 |-- V3: double (nullable = true)
 |-- V4: double (nullable = true)
 |-- V5: double (nullable = true)
 |-- V6: double (nullable = true)
 |-- V7: double (nullable = true)
 |-- V8: double (nullable = true)
 |-- V9: double (nullable = true)
 |-- V10: double (nullable = true)
 |-- V11: double (nullable = true)
 |-- V12: double (nullable = true)
 |-- V13: double (nullable = true)
 |-- V14: double (nullable = true)
 |-- V15: double (nullable = true)
 |-- V16: double (nullable = true)
 |-- V17: double (nullable = true)
 |-- V18: double (nullable = true)
 |-- V19: double (nullable = true)
 |-- V20: double (nullable = true)
 |-- V21: double (nullable = true)
 |-- V22: double (nullable = true)
 |-- V23: double (nullable = true)
 |-- V24: double (nullable = true)
 |-- V25: double (nullable = true)
 |-- V26: double (nullable = true)
 |-- V27: double (nullable = true)
 |-- V28: double (nulla

## Feature Engineering

In [12]:
feature_df = (

    parsed_df

    .withColumn(
        "Large_Transaction",
        when(col("Amount") > 200, 1).otherwise(0)
    )

    .withColumn(
        "Log_Amount",
        log1p(col("Amount"))
    )

    .withColumn(
        "Amount_Level",
        when(col("Amount") < 10, 0)
        .when(col("Amount") < 100, 1)
        .when(col("Amount") < 500, 2)
        .otherwise(3)
    )

)

## Select Features in Model Order

In [13]:
feature_columns = list(model.feature_names_in_)

prediction_df = feature_df.select(

    "transaction_id",

    "event_timestamp",

    *feature_columns

)

## Verify Columns

In [14]:
prediction_df.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- event_timestamp: string (nullable = true)
 |-- Time: double (nullable = true)
 |-- V1: double (nullable = true)
 |-- V2: double (nullable = true)
 |-- V3: double (nullable = true)
 |-- V4: double (nullable = true)
 |-- V5: double (nullable = true)
 |-- V6: double (nullable = true)
 |-- V7: double (nullable = true)
 |-- V8: double (nullable = true)
 |-- V9: double (nullable = true)
 |-- V10: double (nullable = true)
 |-- V11: double (nullable = true)
 |-- V12: double (nullable = true)
 |-- V13: double (nullable = true)
 |-- V14: double (nullable = true)
 |-- V15: double (nullable = true)
 |-- V16: double (nullable = true)
 |-- V17: double (nullable = true)
 |-- V18: double (nullable = true)
 |-- V19: double (nullable = true)
 |-- V20: double (nullable = true)
 |-- V21: double (nullable = true)
 |-- V22: double (nullable = true)
 |-- V23: double (nullable = true)
 |-- V24: double (nullable = true)
 |-- V25: double (nullable = true)


## Create Prediction Function

In [15]:
def predict_batch(batch_df, batch_id):

    pdf = batch_df.toPandas()

    if pdf.empty:
        return

    # Preserve metadata
    metadata_df = pdf[
        [
            "transaction_id",
            "event_timestamp"
        ]
    ].copy()

    # Features for prediction
    X = pdf[model.feature_names_in_]

    # Predictions
    predictions = model.predict(X)

    probabilities = model.predict_proba(X)[:, 1]

    # Final output
    result = metadata_df.copy()

    result["Amount"] = pdf["Amount"]

    result["Prediction"] = predictions

    result["Status"] = result["Prediction"].map({
        0: "✅ Genuine",
        1: "🚨 Fraud"
    })

    result["Fraud Probability"] = probabilities.round(6)

    result["Risk Level"] = pd.cut(
        result["Fraud Probability"],
        bins=[-0.01, 0.20, 0.50, 0.80, 1.00],
        labels=[
            "🟢 Low",
            "🟡 Medium",
            "🟠 High",
            "🔴 Critical"
        ]
    )

    print("\n")
    print("=" * 120)
    print(f"Batch {batch_id}")
    print("=" * 120)

    print(
        result[
            [
                "transaction_id",
                "Amount",
                "Fraud Probability",
                "Risk Level",
                "Status"
            ]
        ]
    )

    frauds = result[result["Prediction"] == 1]

    if not frauds.empty:

        print("\n")
        print("🚨" * 25)
        print("🚨 LIVE FRAUD ALERT 🚨")
        print("🚨" * 25)

        print(
            frauds[
                [
                    "transaction_id",
                    "Amount",
                    "Fraud Probability",
                    "Risk Level"
                ]
            ]
        )

    print("\n")
    print(f"Processed {len(result)} transaction(s)")

## Start Streaming Prediction

In [16]:
query = (

    prediction_df.writeStream

    .foreachBatch(predict_batch)

    .outputMode("append")

    .option(
        "checkpointLocation",
        "../artifacts/checkpoints/realtime_prediction"
    )

    .start()

)

26/07/11 22:41:23 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


## Streaming Status

In [17]:
print(query.status)

{'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


## Active Streams

In [18]:
spark.streams.active

## Keep Alive

In [ ]:
query.awaitTermination()

26/07/11 22:41:23 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.




Batch 101
                         transaction_id  Amount  Fraud Probability Risk Level     Status
0  5e033906-918b-48e2-8e33-ec8e74407ed1  149.62                0.0      🟢 Low  ✅ Genuine


Processed 1 transaction(s)




Batch 102
                         transaction_id  Amount  Fraud Probability Risk Level     Status
0  5196f9ec-bd3d-434a-9dee-15f726828b80    2.69                0.0      🟢 Low  ✅ Genuine


Processed 1 transaction(s)


Batch 103
                         transaction_id  Amount  Fraud Probability Risk Level     Status
0  f383a807-3a5a-40c7-82b7-64b91c43557b  378.66                0.0      🟢 Low  ✅ Genuine


Processed 1 transaction(s)


Batch 104
                         transaction_id  Amount  Fraud Probability Risk Level     Status
0  d4f80622-4314-4a19-a02e-1cba0ec9ead1   123.5                0.0      🟢 Low  ✅ Genuine


Processed 1 transaction(s)


Batch 105
                         transaction_id  Amount  Fraud Probability Risk Level     Status
0  69df7c9e-ce2f-453e-a464-91ad4c52a1c2   69.99                0.0      🟢 Low  ✅ Genuine


Processed 1 transaction(s)


Batch 106
                         transaction_id  Amount  Fraud Probability Risk Level     Status
0  d3e6cb80-8b35-4bd0-a